In [1]:
# ============================================================
# CELL 0 — BOOTSTRAP  (identical across all pipeline notebooks)
# Finds the repo root (folder containing .env), puts src/ on the import
# path, loads shared config + download-log helpers. Machine-agnostic:
# paths come from .env via config.py, never hardcoded.
# ============================================================
import sys                                     # to modify the module search path at runtime
from pathlib import Path                        # portable path handling across Mac/PC

# Walk up from the CWD until the folder containing '.env' (the repo root) is found —
# this is what lets the same notebook run on any machine without editing paths.
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists())
sys.path.insert(0, str(_root / 'src'))          # make 'import config' / 'import download_log' resolve

from config import *                            # PROJECT_ROOT, RAW_DIR, PROCESSED_DIR, CURRENT_YEAR, ...
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime                   # for any runtime date handling
import pandas as pd                             # primary data-handling library

log = load_log()                                # load the download-log ledger

# --- Verify the bootstrap resolved correctly before proceeding ---
print("PROJECT_ROOT :", PROJECT_ROOT)
print("RAW_DIR      :", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("CURRENT_YEAR :", CURRENT_YEAR)

PROJECT_ROOT : C:\Users\mjbou\governance-framework
RAW_DIR      : C:\Users\mjbou\governance-framework\data\raw
PROCESSED_DIR: C:\Users\mjbou\governance-framework\data\processed
CURRENT_YEAR : 2026


# Notebook 38 — World Bank Bank Regulation & Supervision Survey (BRSS)

**Concept 9 (financial-sector regulatory & supervisory quality) — banking-supervision leg. SUPPLEMENTARY tier.**

Builds a transparent, construct-aligned **banking regulatory/supervisory stringency score** from the
World Bank BRSS (5th wave, 2019; reference year 2016; 160 jurisdictions). Fills the banking leg that
FATF (AML/CFT) does not cover.

**Method — bespoke construct-aligned select-and-score (NOT a reproduction of the published BCL indices):**
curated comparable, high-coverage (≥80%), directional questions grouped into 9 sub-constructs
(supervisory power, independence, capital stringency, private monitoring, resolution, provisioning,
liquidity, macroprudential, supervisory capacity), each normalized to a sub-index then averaged.
Activity Restrictions deliberately **excluded** (contested directionality — more restriction ≠ better).

**Source & currency:** frozen 2019 wave (irregular survey: 2001/03/07/11/19). Reference year (2016)
derived from question codes, not hardcoded. **Update = manual check for a 6th wave** (see instructions doc).
CC-BY-4.0 (commercial use OK). Fetch auto-discovers the latest `.xlsx` from the WB catalog page.

In [2]:
# ============================================================
# CELL 2 — CONFIG
# Source id, paths, and the auto-discover fetch target. The only fetch constant is
# the PERMANENT WB Data Catalog dataset URL (it's the dataset identifier, not a
# versioned file link — so it won't rot like the version-pinned URL that 404'd);
# Cell 4 scrapes it for the latest BRSS *.xlsx. No hardcoded vintage — the 2016
# reference year is derived from the data in Cell 5.
# ============================================================
import os

SOURCE_ID = "WB_BRSS"   # World Bank Bank Regulation & Supervision Survey (distinct — confirmed not in registry)

# --- Fetch: auto-discover from the permanent catalog page (Chinn-Ito pattern) --------
# ⚠️ MANUAL-MAINTENANCE CONSTANT: dataset's permanent catalog URL. Stable (dataset id,
#    not a versioned file link). Update ONLY if WB restructures its catalog. Cell 4
#    scrapes this for the newest BRSS *.xlsx and downloads if not already cached.
BRSS_CATALOG_URL = "https://datacatalog.worldbank.org/search/dataset/0038632/bank-regulation-and-supervision-survey"

# --- Paths --------------------------------------------------------------------------
RAW_XLSX   = os.path.join(RAW_DIR,       "wb_brss_2019.xlsx")   # cached source workbook (data/raw)
OUTPUT_CSV = os.path.join(PROCESSED_DIR, "wb_brss_clean.csv")   # pipeline output (cross-section)

# --- Confirm setup ------------------------------------------------------------------
print("SOURCE_ID :", SOURCE_ID)
print("catalog   :", BRSS_CATALOG_URL)
print("RAW_XLSX  :", RAW_XLSX, "\n            (cached already:", os.path.exists(RAW_XLSX), ")")
print("OUTPUT_CSV:", OUTPUT_CSV)
print(f"'{SOURCE_ID}' in download_log:", SOURCE_ID in list(log["source_id"]), "(expect False on first run)")

SOURCE_ID : WB_BRSS
catalog   : https://datacatalog.worldbank.org/search/dataset/0038632/bank-regulation-and-supervision-survey
RAW_XLSX  : C:\Users\mjbou\governance-framework\data\raw\wb_brss_2019.xlsx 
            (cached already: True )
OUTPUT_CSV: C:\Users\mjbou\governance-framework\data\processed\wb_brss_clean.csv
'WB_BRSS' in download_log: False (expect False on first run)


In [3]:
# ============================================================
# CELL 3 — INCLUDED_QUESTIONS  (curation map; final sign-off point)
#
# ⚠️ MANUAL-MAINTENANCE CONSTANT — the codes below are the SURVEY-V (2019 wave) question
#    NUMBERS. On a NEW BRSS wave the survey may renumber/reword questions, so this map must be
#    RE-VALIDATED against the new questionnaire (check each code still exists AND still means the
#    same thing/direction). The YEAR suffix is NOT hardcoded — Cell 5 auto-derives the latest year
#    per code and FAILS LOUDLY listing any code that no longer resolves. See instructions_data_maintenance.md.
#
# Transparent construct-aligned scoring set (NOT the published BCL indices). 9 sub-constructs,
# all clear-directional (Activity Restrictions dropped: contested directionality).
#   binary : (base_code, sign)                Yes->1 / No->0, then * sign
#   numeric: (base_code, sign)                value min-max normalized across countries, then * sign
#   block  : (name, [base_codes], mode, sign) mode "fraction" = #Yes/n ; mode "any" = (any Yes)->1
#   sign: +1 = higher/Yes -> STRONGER ; -1 = REVERSE.  [REV] items flagged inline.
#
# Provisioning/macroprudential trimmed after per-item evidence showed de-jure "stringency inflation"
# (prescriptive rules penalizing IFRS-9 regimes; breadth-of-tools counting). See framework_decisions.md.
# ============================================================
INCLUDED_QUESTIONS = {
  "supervisory_power": {"binary": [
      ("Q12_5",  +1),  # force org-structure change
      ("Q12_15", +1),  # acts when infraction found
      ("Q12_16", +1),  # mandatory actions on infraction
      ("Q11_2",  +1),  # enforcement actions made public
      ("Q11_4",  +1),  # early-intervention / PCA framework
      ("Q12_3",  -1)]},# [REV] deposit-takers outside prudential supervision

  "supervisory_independence": {"binary": [
      ("Q12_8",  +1),  # head appointed on external panel recommendation
      ("Q12_9",  +1),  # head has fixed term
      ("Q12_12", -1),  # [REV] staff personally liable -> less protection
      ("Q12_13", -1)]},# [REV] agency liable -> less protection

  "capital_stringency": {
      "numeric": [("Q3_3_1", +1),    # min REQUIRED risk-based capital ratio
                  ("Q3_3_3", +1)],   # min REQUIRED Tier-1 ratio
      "binary":  [("Q3_6_1", +1),    # ICAAP required
                  ("Q3_7",   +1),    # power to require higher capital
                  ("Q3_8",   +1),    # power to require add-on capital for risk
                  ("Q3_13",  +1),    # conservation buffer
                  ("Q3_14",  +1)],   # countercyclical buffer
      "block":   [("tier1_deductions",  # Basel-III deductions from Tier-1; count IS meaningful
                   [f"Q3_20_3{x}" for x in "abcdefghi"], "fraction", +1)]},

  "private_monitoring": {"binary": [
      ("Q5_1",  +1), ("Q5_6", +1), ("Q5_7", +1), ("Q5_8", +1),      # audit required / ISA / independence / public
      ("Q5_10", +1), ("Q5_10_1", +1), ("Q5_11", +1),                # auditor -> supervisor reporting
      ("Q10_1", +1), ("Q10_6", +1), ("Q10_9", +1), ("Q10_11", +1)]},# consolidation/submission/liability/ratings

  "resolution_regime": {"binary": [
      ("Q11_5", +1), ("Q11_18", +1), ("Q11_20", +1), ("Q11_21", +1),
      ("Q11_22", +1), ("Q11_23", +1), ("Q11_24", +1)]},

  "provisioning": {"binary": [
      ("Q9_1", +1), ("Q9_7", +1), ("Q9_8", +1), ("Q9_11", +1),   # dropped Q9_9, Q9_12: prescriptive rules that
                                                                 #   penalize IFRS-9 / expected-loss regimes
      ("Q9_5", -1), ("Q9_6", -1)]},                              # [REV] lax income recognition / immediate upgrade

  "liquidity_concentration": {"binary": [
      ("Q7_1", +1), ("Q7_3", +1), ("Q7_7", +1), ("Q7_8", +1), ("Q6_7", +1)]},

  "macroprudential": {
      "binary": [
          ("Q12_25", +1),                       # has macroprudential mandate
          ("Q12_27", +1),                       # conducts stress tests
          ("Q12_34", +1), ("Q12_35", +1),       # SIFIs supervised differently / measures systemic contribution
          ("Q12_37", +1), ("Q12_38", +1)],      # SIFI resolution / monitors interconnectedness
                                                 # dropped Q12_26 (FS report): capability item, but +50 gap looks like
                                                 #   a question-design artifact (FS analysis often published elsewhere)
      "block": [("borrower_based_caps",          # LTV/DTI/down-pmt/DTV collapsed to ONE "has toolkit" point
                 ["Q12_29","Q12_30","Q12_31","Q12_33"], "any", +1)]},

  "supervisory_capacity": {"numeric": [
      ("Q12_41", +1), ("Q12_42", +1)]},          # % supervisors w/ college / post-grad degree
}

# --- Summary for sign-off ---------------------------------------------------
_tot = 0
print(f"{'sub-construct':<26}{'items':>6}   (reverse-coded)")
for k, v in INCLUDED_QUESTIONS.items():
    n = len(v.get("binary", [])) + len(v.get("numeric", [])) + len(v.get("block", []))
    _tot += n
    revs = [c for grp in ("binary","numeric") for c,s in v.get(grp,[]) if s < 0]
    print(f"  {k:<24}{n:>6}   {('REV: '+', '.join(revs)) if revs else ''}")
print(f"\n  TOTAL scored items (each block counts as 1): {_tot} across {len(INCLUDED_QUESTIONS)} sub-constructs")
print("  blocks: tier1_deductions (fraction, 9 sub-items) | borrower_based_caps (any, 4 sub-items)")

sub-construct              items   (reverse-coded)
  supervisory_power            6   REV: Q12_3
  supervisory_independence     4   REV: Q12_12, Q12_13
  capital_stringency           8   
  private_monitoring          11   
  resolution_regime            7   
  provisioning                 6   REV: Q9_5, Q9_6
  liquidity_concentration      5   
  macroprudential              7   
  supervisory_capacity         2   

  TOTAL scored items (each block counts as 1): 56 across 9 sub-constructs
  blocks: tier1_deductions (fraction, 9 sub-items) | borrower_based_caps (any, 4 sub-items)


In [4]:
# ============================================================
# CELL 4 — AUTO-DISCOVER FETCH  (no hardcoded file URL; scrapes the permanent catalog page)
# Finds BRSS 'public-release' .xlsx links on BRSS_CATALOG_URL, picks the NEWEST by the year in the
# filename, then:
#   - cache absent           -> download the newest
#   - cache present + newer file on page -> LOUD new-wave alert (do NOT auto-adopt: a new wave can
#                               renumber questions, so Cell-3 must be re-validated first)
#   - else                   -> use the cache
# Discovery degrades gracefully to the cache if the page can't be scraped (fails safe).
# ============================================================
import re, requests, shutil

CACHE_YEAR = 2021   # ⚠️ MANUAL-MAINTENANCE CONSTANT: file-year of the wave our cache + Cell-3 curation
                    #    target (2019 wave, 2021-updated release). Bump ONLY when deliberately adopting a
                    #    NEW wave, AFTER re-validating INCLUDED_QUESTIONS. Drives new-wave detection below.

def discover_brss_xlsx(catalog_url):
    r = requests.get(catalog_url, headers=BROWSER_HEADERS, verify=SSL_VERIFY, timeout=60)
    r.raise_for_status()
    urls = sorted(set(re.findall(
        r'https://datacatalogfiles\.worldbank\.org/[^\s"\'<>]+?brss-public-release\.xlsx', r.text)))
    if not urls:
        raise RuntimeError("no BRSS public-release .xlsx links found (page structure changed?)")
    yr = lambda u: max([int(y) for y in re.findall(r'20\d{2}', u.rsplit('/', 1)[-1])] or [0])
    return max(urls, key=yr), yr(max(urls, key=yr)), urls

# --- try discovery; degrade to cache if the page can't be read ----------------------
try:
    newest_url, newest_yr, all_found = discover_brss_xlsx(BRSS_CATALOG_URL)
    print(f"Discovered {len(all_found)} release file(s); newest embeds year {newest_yr}:")
    for u in all_found:
        print("   ", ("-> " if u == newest_url else "   ") + u.rsplit('/', 1)[-1])
    discovery_ok = True
except Exception as e:
    print("Auto-discover FAILED (will fall back to cache):", e)
    discovery_ok = False

# --- cache-first + first-run download + new-wave alert ------------------------------
if not os.path.exists(RAW_XLSX):
    if not discovery_ok:
        raise RuntimeError("No cached file AND discovery failed. Manually download the latest BRSS "
                           "public-release .xlsx from BRSS_CATALOG_URL to RAW_XLSX, then re-run.")
    print(f"\nNo local cache -> downloading {newest_url.rsplit('/', 1)[-1]}")
    with requests.get(newest_url, headers=BROWSER_HEADERS, verify=SSL_VERIFY, timeout=120, stream=True) as resp:
        resp.raise_for_status()
        with open(RAW_XLSX, "wb") as fh:
            shutil.copyfileobj(resp.raw, fh)
    print("   downloaded:", os.path.getsize(RAW_XLSX), "bytes")
elif discovery_ok and newest_yr > CACHE_YEAR:
    print(f"\n*** NEW BRSS WAVE DETECTED (page file-year {newest_yr} > cached {CACHE_YEAR}). ***")
    print("*** Do NOT auto-adopt: re-validate Cell-3 INCLUDED_QUESTIONS against the new questionnaire,")
    print("*** then bump CACHE_YEAR and delete the old cache to re-download. Using cached file for now.")
else:
    print(f"\nUsing cached file (cache year {CACHE_YEAR}; no newer wave found):", os.path.basename(RAW_XLSX))

Discovered 2 release file(s); newest embeds year 2021:
       survey-20191104-brss-public-release.xlsx
    -> 2021_04_26_brss-public-release.xlsx

Using cached file (cache year 2021; no newer wave found): wb_brss_2019.xlsx


In [5]:
# ============================================================
# CELL 5 — PARSE WORKBOOK: resolve year, guard codes, transpose to country rows, attach ISO3
# For each base code in INCLUDED_QUESTIONS: locate its row in the matching topic sheet by matching
# '<base>_<YYYY>' and taking the LATEST year (no year hardcoded). FAILS LOUDLY listing any base code
# that doesn't resolve (catches a wave renumbering). Then melt the transposed sheets (questions=rows,
# countries=cols) into a country x code raw frame and attach ISO3 from the 'General' sheet.
# ============================================================
import re

# --- every base code in the curation map (de-duped, order-preserving) ---------------
def all_codes(m):
    for _, g in m.items():
        for grp in ("binary", "numeric"):
            for c, _ in g.get(grp, []): yield c
        for _, codelist, *_ in g.get("block", []):
            for c in codelist: yield c
CODES = list(dict.fromkeys(all_codes(INCLUDED_QUESTIONS)))

def sheet_of(base):                       # 'Q12_5'->'12' ; 'Q3_20_3a'->'03'
    return f"{int(base.split('_')[0][1:]):02d}"

_sheets = {}
def load_sheet(sh):
    if sh not in _sheets:
        df = pd.read_excel(RAW_XLSX, sheet_name=sh, header=0)
        idx = df[df.columns[0]].astype(str).str.strip()      # the 'Index' (question code) column
        ctry = [str(c).strip() for c in df.columns[2:]]       # cols 2+ = jurisdiction columns
        df = df.set_axis([df.columns[0], df.columns[1], *ctry], axis=1)
        _sheets[sh] = (df, idx, ctry)
    return _sheets[sh]

def resolve_latest(base, idx_series):     # '<base>_<YYYY>' -> exact index str with the max year
    pat = re.compile(rf"^{re.escape(base)}_(20\d{{2}})$")
    hits = []
    for s in idx_series:
        s = str(s)                        # defensive: blank parent rows come through as NaN (float)
        m = pat.match(s)
        if m:
            hits.append((int(m.group(1)), s))
    return max(hits)[1] if hits else None

# --- master jurisdiction list + ISO3 from 'General' ---------------------------------
gen = pd.read_excel(RAW_XLSX, sheet_name="General", header=0)
gen_name = gen["Country Name"].astype(str).str.strip()
name2iso = dict(zip(gen_name, gen["Country Code"].astype(str).str.strip()))
# ⚠️ MANUAL-MAINTENANCE CONSTANT: ISO3 overrides for jurisdictions the file leaves blank/non-standard.
#    Extend only if Cell-5 reports a new "no ISO3" jurisdiction. (Curaçao -> CUW; file ships it blank.)
ISO3_OVERRIDES = {"Curaçao": "CUW"}
name2iso.update({k: v for k, v in ISO3_OVERRIDES.items() if k in gen_name.values})
MASTER = list(gen_name)

# --- guard: topic-sheet country columns align with General (align by NAME, not position) ---
_, _, ctry12 = load_sheet("12")
mismatch = set(ctry12) ^ set(MASTER)
if mismatch:
    print("WARNING: country-name mismatch sheet12 vs General (will NaN on reindex):", list(mismatch)[:10])

# --- resolve each code; pull its per-country response series ------------------------
resolved, unresolved, cols = {}, [], {}
for base in CODES:
    df, idx, ctry = load_sheet(sheet_of(base))
    actual = resolve_latest(base, idx)
    resolved[base] = actual
    if actual is None:
        unresolved.append(base); continue
    ser = df.loc[idx.values == actual].iloc[0][ctry]     # response per country-name
    cols[base] = pd.Series(ser.values, index=ctry).reindex(MASTER)

# --- FAIL LOUD on any unresolved code (renumbering / typo) --------------------------
if unresolved:
    print("UNRESOLVED base codes:")
    for b in unresolved: print("   ", b, "-> expected in sheet", sheet_of(b))
    raise AssertionError(f"{len(unresolved)} code(s) unresolved — re-validate INCLUDED_QUESTIONS "
                         f"against the current questionnaire (Cell-3 note).")

# --- assemble country x code raw frame, attach ISO3 --------------------------------
raw = pd.DataFrame(cols, index=MASTER)
raw.insert(0, "country_code", [name2iso.get(n) for n in MASTER])
raw.insert(1, "country_name", MASTER)
no_iso = raw[raw["country_code"].isna() | (raw["country_code"] == "")]

print(f"Resolved ALL {len(CODES)} base codes (year auto-derived).")
print("Year mapping sample:", {b: resolved[b] for b in list(CODES)[:5]})
print("Raw frame shape:", raw.shape, "-> jurisdictions x (2 id cols + code cols)")
print("Jurisdictions with no ISO3:", list(no_iso["country_name"]) if len(no_iso) else "none")
print("Per-code non-missing coverage (min / median / max %):",
      *[f"{v:.0f}" for v in [raw[list(cols)].notna().mean().min()*100,
                             raw[list(cols)].notna().mean().median()*100,
                             raw[list(cols)].notna().mean().max()*100]])

Resolved ALL 67 base codes (year auto-derived).
Year mapping sample: {'Q12_5': 'Q12_5_2016', 'Q12_15': 'Q12_15_2016', 'Q12_16': 'Q12_16_2016', 'Q11_2': 'Q11_2_2016', 'Q11_4': 'Q11_4_2016'}
Raw frame shape: (161, 69) -> jurisdictions x (2 id cols + code cols)
Jurisdictions with no ISO3: none
Per-code non-missing coverage (min / median / max %): 81 98 99


In [6]:
# ============================================================
# CELL 6 — SCORING: normalize -> sign -> 9 sub-construct scores -> weighted de-jure STRINGENCY.
# NO coverage penalty (that conflated "unanswered" with "weak"). Instead: emit the score for all,
# plus coverage + a reliability FLAG so low-coverage entries are excluded DOWNSTREAM, not deflated.
# Output is a DE JURE regulatory-stringency score (rules-on-paper), NOT supervisory effectiveness —
# validated against Anginer et al. (2019), who document the same HI/DEV pattern on this dataset.
# ============================================================
import numpy as np

WEIGHTS = {"supervisory_power":2,"supervisory_independence":2,"private_monitoring":2,
           "resolution_regime":2,"macroprudential":2,"capital_stringency":1,"provisioning":1,
           "liquidity_concentration":1,"supervisory_capacity":1}

# ⚠️ METHODOLOGY CONSTANT: coverage below which a jurisdiction's score is flagged UNRELIABLE
#    (score still emitted for transparency, but brss_reliable=False -> exclude downstream).
#    0.70 sits in the gap between the sparse tail (<=60%) and the main mass (>=85%).
#    REVISIT post-v1 (low priority) — see framework_decisions.md.
RELIABILITY_MIN_COVERAGE = 0.70

def norm_binary(s):
    v = s.astype(str).str.strip().str.lower()
    out = pd.Series(np.nan, index=s.index, dtype="float64")
    out[v == "yes"] = 1.0; out[v == "no"] = 0.0
    return out
def norm_numeric(s):
    x = pd.to_numeric(s, errors="coerce"); lo, hi = x.min(), x.max()
    return (x - lo) / (hi - lo) if hi > lo else x * 0.0
def norm_block(codes, mode):
    b = pd.concat([norm_binary(raw[c]) for c in codes], axis=1)
    sc = (b.fillna(0).sum(axis=1) > 0).astype(float) if mode == "any" else b.fillna(0).sum(axis=1)/len(codes)
    sc[b.isna().all(axis=1)] = np.nan
    return sc

# --- normalize items, apply sign, mean into sub-constructs (mean of ANSWERED items) --
sub_scores, all_items = {}, []
for sub, groups in INCLUDED_QUESTIONS.items():
    cols = []
    for code, sign in groups.get("binary", []):
        v = norm_binary(raw[code]);  cols.append(v if sign > 0 else 1 - v)
    for code, sign in groups.get("numeric", []):
        v = norm_numeric(raw[code]); cols.append(v if sign > 0 else 1 - v)
    for _, codes, mode, sign in groups.get("block", []):
        v = norm_block(codes, mode); cols.append(v if sign > 0 else 1 - v)
    M = pd.concat(cols, axis=1); all_items.extend(cols)
    sub_scores[sub] = M.mean(axis=1, skipna=True)
S = pd.DataFrame(sub_scores)
coverage = pd.concat(all_items, axis=1).notna().mean(axis=1)

# --- weighted de-jure stringency (NO coverage penalty); renormalize weights over present subs ---
w = pd.Series(WEIGHTS)[S.columns]
stringency = S.mul(w, axis=1).sum(axis=1) / S.notna().mul(w, axis=1).sum(axis=1)

# --- assemble ------------------------------------------------------------------------
scored = pd.DataFrame({"country_code": raw["country_code"], "country_name": raw["country_name"]})
for c in S.columns: scored[c] = S[c].round(4)
scored["brss_regstringency"] = stringency.round(4)     # de-jure regulatory stringency (headline)
scored["brss_coverage"]      = coverage.round(3)
scored["brss_reliable"]      = coverage >= RELIABILITY_MIN_COVERAGE

# --- report RELIABLE-subset ranking --------------------------------------------------
rel = scored[scored["brss_reliable"]]
print(f"Reliable (coverage >= {RELIABILITY_MIN_COVERAGE:.0%}): {len(rel)}/{len(scored)}"
      f"   | flagged unreliable: {(~scored['brss_reliable']).sum()}")
print("Flagged:", ", ".join(f"{r.country_name}({r.brss_coverage:.0%})" for r in scored[~scored['brss_reliable']].itertuples()))
print("\nSub-construct mean [min,max] (reliable only):")
for c in S.columns:
    rc = rel[c]; print(f"  {c:<26} {rc.mean():.3f} [{rc.min():.3f}, {rc.max():.3f}]")
srt = rel.sort_values("brss_regstringency", ascending=False)
print(f"\nDe-jure stringency (reliable) mean {srt.brss_regstringency.mean():.3f} range [{srt.brss_regstringency.min():.3f}, {srt.brss_regstringency.max():.3f}]")
print("\nTop 10:\n",    srt[["country_name","brss_regstringency","brss_coverage"]].head(10).to_string(index=False))
print("\nBottom 10:\n", srt[["country_name","brss_regstringency","brss_coverage"]].tail(10).to_string(index=False))

Reliable (coverage >= 70%): 155/161   | flagged unreliable: 6
Flagged: Comoros(20%), Congo, Dem. Rep.(16%), Eswatini(30%), Euro Area(61%), Montserrat(46%), Turks and Caicos Islands(70%)

Sub-construct mean [min,max] (reliable only):
  supervisory_power          0.736 [0.250, 1.000]
  supervisory_independence   0.624 [0.000, 1.000]
  capital_stringency         0.573 [0.073, 0.920]
  private_monitoring         0.734 [0.364, 1.000]
  resolution_regime          0.424 [0.000, 1.000]
  provisioning               0.755 [0.000, 1.000]
  liquidity_concentration    0.628 [0.200, 1.000]
  macroprudential            0.622 [0.000, 1.000]
  supervisory_capacity       0.719 [0.085, 1.000]

De-jure stringency (reliable) mean 0.639 range [0.368, 0.854]

Top 10:
 country_name  brss_regstringency  brss_coverage
     Nigeria              0.8542          0.946
       Qatar              0.8437          0.929
    Slovenia              0.7992          0.964
       Italy              0.7908          0.982
  Ba

In [7]:
# ============================================================
# CELL 7 — STRUCTURAL INTEGRITY GUARDS
# Structural checks only (domains, nulls, internal consistency) — NOT vintage-specific counts,
# so nothing here needs editing on a new wave. Collects ALL violations, reports together, then
# fails safe before Cell 8 writes anything. Row counts printed, never asserted.
# ============================================================
problems = []

SUBS = list(INCLUDED_QUESTIONS.keys())   # the 9 sub-construct columns

# (1) identifier integrity
n_noiso = int(scored["country_code"].isna().sum() + (scored["country_code"].astype(str).str.strip()=="").sum())
if n_noiso: problems.append(f"{n_noiso} row(s) with no ISO3")
dups = [c for c,n in scored["country_code"].value_counts().items() if n>1 and str(c).strip()]
if dups: problems.append(f"duplicate ISO3: {dups}")

# (2) score domains: every sub-score and the headline must be in [0,1] (or NaN)
for col in SUBS + ["brss_regstringency"]:
    bad = scored[(scored[col].notna()) & ((scored[col] < 0) | (scored[col] > 1))]
    if len(bad): problems.append(f"{col}: {len(bad)} value(s) outside [0,1]")

# (3) coverage in [0,1]; reliable flag is boolean and consistent with the threshold
if not scored["brss_coverage"].between(0,1).all(): problems.append("coverage outside [0,1]")
mis = scored[scored["brss_reliable"] != (scored["brss_coverage"] >= RELIABILITY_MIN_COVERAGE)]
if len(mis): problems.append(f"{len(mis)} row(s): brss_reliable inconsistent with coverage threshold")

# (4) headline must exist wherever >=1 sub-score exists (no silent all-NaN survivors)
should = scored[SUBS].notna().any(axis=1)
missing_head = scored[should & scored["brss_regstringency"].isna()]
if len(missing_head): problems.append(f"{len(missing_head)} row(s) have sub-scores but no headline")

# (5) reliable rows should have high coverage AND most sub-scores present (sanity)
rel = scored[scored["brss_reliable"]]
thin_rel = rel[rel[SUBS].notna().sum(axis=1) < 7]
if len(thin_rel): problems.append(f"{len(thin_rel)} 'reliable' row(s) have <7 of 9 sub-scores: {list(thin_rel['country_name'])}")

# --- report all, then fail safe -----------------------------------------------------
if problems:
    print("INTEGRITY VIOLATIONS (nothing will be written):")
    for p in problems: print("  -", p)
    raise AssertionError(f"{len(problems)} violation(s) — see above.")
print("All structural integrity guards PASSED.\n")
print("Rows:", len(scored), "| reliable:", int(scored['brss_reliable'].sum()),
      "| flagged:", int((~scored['brss_reliable']).sum()))
print("Headline present:", int(scored['brss_regstringency'].notna().sum()),
      "| sub-score cols:", len(SUBS))
print("\nColumns to be written:", ["country_code","country_name"] + SUBS +
      ["brss_regstringency","brss_coverage","brss_reliable"])

All structural integrity guards PASSED.

Rows: 161 | reliable: 155 | flagged: 6
Headline present: 161 | sub-score cols: 9

Columns to be written: ['country_code', 'country_name', 'supervisory_power', 'supervisory_independence', 'capital_stringency', 'private_monitoring', 'resolution_regime', 'provisioning', 'liquidity_concentration', 'macroprudential', 'supervisory_capacity', 'brss_regstringency', 'brss_coverage', 'brss_reliable']


In [8]:

# ============================================================
# CELL 8 — WRITE CLEAN OUTPUT + REGISTER IN DOWNLOAD_LOG
# Writes wb_brss_clean.csv in the house CROSS-SECTION convention: country_code-keyed, NO 'year',
# NO 'country_name' (re-attached downstream at merge — matches rti/pefa/fatf). Registers WB_BRSS
# in download_log with vintage derived from the data (question-code year), NOT hardcoded.
# Does NOT touch source_registry.csv — that's owned by notebook 02 (separate cell provided after).
# ============================================================
SUBS = list(INCLUDED_QUESTIONS.keys())

# --- vintage derived from the data (question-code year), not hardcoded ---------------
# The response reference year is embedded in the resolved codes (Cell 5); take the latest.
DATA_AS_OF = str(max(int(y) for code in resolved.values() if code
                     for y in [code.rsplit("_", 1)[-1]] if y.isdigit()))   # e.g. '2016'

# --- assemble output: country_code + 9 sub-scores + headline + coverage + reliable ---
OUTPUT_COLS = ["country_code"] + SUBS + ["brss_regstringency", "brss_coverage", "brss_reliable"]
out = scored[OUTPUT_COLS].copy()
out = out[out["country_code"].notna() & (out["country_code"].astype(str).str.strip() != "")]  # drop no-ISO3 (none expected)
out.to_csv(OUTPUT_CSV, index=False)
print("Wrote  :", OUTPUT_CSV)
print("Shape  :", out.shape, "(rows, cols)")
print("Columns:", list(out.columns))
print("\nSample (reliable, top of ranking):")
print(out[out["brss_reliable"]].sort_values("brss_regstringency", ascending=False).head(3).to_string(index=False))

# --- register in download_log; vintage from data, not typed --------------------------
today_str = datetime.today().strftime("%Y-%m-%d")
update_entry(
    SOURCE_ID,                                       # WB_BRSS
    last_successful_download_date=today_str,
    data_as_of_date=f"{DATA_AS_OF} (BRSS 5th wave, 2019 survey; reference year {DATA_AS_OF})",
    local_filename=os.path.basename(OUTPUT_CSV),     # wb_brss_clean.csv
    latest_available_version="BRSS 5th wave (2019 survey, 2021-updated release)",
    notes=(
        "World Bank Bank Regulation & Supervision Survey (5th wave, 2019; reference year 2016). "
        "SUPPLEMENTARY tier — Concept 9 banking-supervision leg (complements FATF AML/CFT). "
        "DE JURE regulatory-STRINGENCY score (rules-on-paper), NOT supervisory effectiveness: advanced "
        "economies mid-pack is CORRECT and validated against Anginer et al. (2019), whose HI/DEV directional "
        "findings this reproduces with zero construct inversions. Transparent construct-aligned select-and-score "
        "(NOT the published BCL indices): 9 sub-constructs (supervisory power/independence, capital stringency, "
        "private monitoring, resolution, provisioning, liquidity, macroprudential, capacity); comparable ≥80%-"
        "coverage items only; Activity Restrictions dropped (contested directionality); provisioning/macropru "
        "trimmed of prescriptive-rule items that penalize IFRS-9/principle-based regimes. Equal-weight within "
        "construct; 5 constructs (power/independence/monitoring/resolution/macropru) weighted 2x in overall. "
        "brss_reliable = coverage>=70% (6 flagged out: Comoros/DRC/Eswatini/Euro Area/Montserrat/Turks&Caicos). "
        "Cross-section snapshot; NO year. FROZEN wave — MANUAL check for a 6th wave (auto-alerted by pipeline); "
        "on a new wave, RE-VALIDATE INCLUDED_QUESTIONS (question renumbering). ISO3 in-file (+Curaçao override). "
        "Auto-discover fetch from WB catalog; CC-BY-4.0. See instructions_data_maintenance.md."
    ),
)
print("\n--- download_log entry ---")
print_entry(SOURCE_ID)

Wrote  : C:\Users\mjbou\governance-framework\data\processed\wb_brss_clean.csv
Shape  : (161, 13) (rows, cols)
Columns: ['country_code', 'supervisory_power', 'supervisory_independence', 'capital_stringency', 'private_monitoring', 'resolution_regime', 'provisioning', 'liquidity_concentration', 'macroprudential', 'supervisory_capacity', 'brss_regstringency', 'brss_coverage', 'brss_reliable']

Sample (reliable, top of ranking):
country_code  supervisory_power  supervisory_independence  capital_stringency  private_monitoring  resolution_regime  provisioning  liquidity_concentration  macroprudential  supervisory_capacity  brss_regstringency  brss_coverage  brss_reliable
         NGA             0.8333                      1.00              0.5913              0.9091             0.6667        1.0000                      0.6           1.0000                 0.950              0.8542          0.946           True
         QAT             0.8333                      0.75              0.9095     